# Netrunner Agenda Density Analysis

In [1]:
# Imports

import pandas as pd
import main as m
import regex as re

Left adjust Markdown tables

In [2]:
%%html
<style>
    table {
        margin-left: 0 !important;
        float: left;
    }
</style>

# Wrangle

In [3]:
df = pd.read_excel('accesses.xlsx')

In Netrunner a corporation deck must contain a total number of cards (deck size) and agenda points according to the following chart

This Dataset represents all possible variations of corporation decks with regard to max/min deck dize, agenda point total, and distribution of one, two, and three point agenda cards and inclution of 'Let them Dream' agendas. This data set was generated using values found in the standard card pool as of (April 21, 2026).

|Deck Size|Agenda Points|
|---|---|
|40 to 44 | 18 or 19 |
|45 to 49 | 20 or 21 |
|50 to 54 | 22 or 23 |

<br>
<br>
<br>
<br>
<br>

## Data Dictionary

|Feature|Definition|
|---|---|
|Agenda Type| "Normal" indicates that there are no Let Them Dream Agendas in the deck "Let Them Dream" indicates that the maximum number of two-point agendas are "Let Them Dream"|
|Deck Size| Count of cards in the deck|
|Agenda Points| Total agenda points used in the deck|
|Agenda Counts| Count of each agenda in the deck by point value \[three-point agendas, two-point agendas, one-point agendas\]|
|Let Them Dream Count| Count of "Let Them Dream" cards in the deck|
|Num Agendas| Count of agendas in deck|
|Average Accesses| Number of unique cards a runner must access, on average, to win the game|

<br>
<br>

**Let them Dream** is an agenda worth two points for the corp, but only one point for the runner

* This card counts as a two point agenda when counting the total Agenda Points in a corporation deck
* This card is represented as a one-point agenda in Agenda Counts and when determining Average Accesses

**Average Accesses** was determined using digital experiments

* Method
    * Generate list representing agenda values of each card in the corporation's deck using zero to represent non-agenda cards
    * Choose a 'card' from the list at random for the runner to 'access'
    * Add the the value of the 'accessed' card to the runners total and remove the card from the list
    * Continue choosing new 'cards' until the runner has accessed 7 points worth of cards and note the number of accesses
    * Repeat the experiment 100_000 times and calculate the average number of accesses across all of the experiments


## Inspection

Data types are as expected

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 992 entries, 0 to 991
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   agenda_type           992 non-null    object
 1   deck_size             992 non-null    int64 
 2   agenda_points         992 non-null    int64 
 3   let_them_dream_count  992 non-null    int64 
 4   agenda_counts         992 non-null    object
 5   num_agendas           992 non-null    int64 
 6   average_accesses      992 non-null    int64 
dtypes: int64(5), object(2)
memory usage: 54.4+ KB


**Deck sizes and agenda point totals are matched to reflect deckbuilding requirements**

In [15]:
deck_sizes = [40, 44, 45, 49, 50, 54]

for deck_size in deck_sizes:

    print(df[['deck_size', 'agenda_points']][df.deck_size == deck_size].drop_duplicates())

    deck_size  agenda_points
0          40             18
70         40             19
     deck_size  agenda_points
144         44             18
214         44             19
     deck_size  agenda_points
288         45             20
368         45             21
     deck_size  agenda_points
454         49             20
534         49             21
     deck_size  agenda_points
620         50             22
710         50             23
     deck_size  agenda_points
806         54             22
896         54             23


Agenda Counts are accurate

In [17]:
df['threes'] = df.agenda_counts.apply(lambda x: int(re.search('\[(\d+),', x).group(1)))
df['twos'] = df.agenda_counts.apply(lambda x: int(re.search(', (\d+),', x).group(1)))
df['ones'] = df.agenda_counts.apply(lambda x: int(re.search(', (\d+)\]', x).group(1)))

df['calc_agenda_points'] = (df['threes'] * 3) + (df['twos'] * 2) + (df['ones']) + (df['let_them_dream_count'])

df[['agenda_counts', 'threes', 'twos', 'ones', 'let_them_dream_count', 'calc_agenda_points', 'agenda_points']]

,agenda_counts,threes,twos,ones,let_them_dream_count,calc_agenda_points,agenda_points
0,"[0, 0, 16]",0,0,16,1,17,18
1,"[0, 0, 15]",0,0,15,1,16,18
2,"[0, 1, 13]",0,1,13,1,16,18
3,"[0, 2, 11]",0,2,11,1,16,18
4,"[0, 3, 9]",0,3,9,1,16,18
...,...,...,...,...,...,...,...
987,"[6, 0, 5]",6,0,5,0,23,23
988,"[6, 1, 3]",6,1,3,0,23,23
989,"[6, 2, 1]",6,2,1,0,23,23
990,"[7, 0, 2]",7,0,2,0,23,23


In [18]:
df.let_them_dream_count.value_counts()

let_them_dream_count
0    496
1    332
2    164
Name: count, dtype: int64

## Explore

The range of average accesses is between 14 and 20 <br>
Agenda density accounts for at most 6 additional/fewer accesses the runner needs to win

In [8]:
df.describe().astype(int)[['average_accesses']].loc[['min','max']]

,average_accesses
min,14
max,20


In [11]:
df[df.deck_size == 40]

,agenda_type,deck_size,agenda_points,let_them_dream_count,agenda_counts,num_agendas,average_accesses
0,Let Them Dream,40,18,1,"[0, 0, 16]",16,17
1,Let Them Dream,40,18,1,"[0, 0, 15]",15,18
2,Let Them Dream,40,18,1,"[0, 1, 13]",14,18
3,Let Them Dream,40,18,1,"[0, 2, 11]",13,18
4,Let Them Dream,40,18,1,"[0, 3, 9]",12,18
...,...,...,...,...,...,...,...
139,Normal,40,19,0,"[4, 3, 1]",8,15
140,Normal,40,19,0,"[5, 0, 4]",9,15
141,Normal,40,19,0,"[5, 1, 2]",8,15
142,Normal,40,19,0,"[5, 2, 0]",7,15


In [ ]:
df

In [ ]:
/df['twos'] = df.agenda_counts.apply(lambda x: re.search(',(\d+),', x))
df['ones'] = df.agenda_counts.apply(lambda x: re.search(',(\d+)\]', x))
df

In [ ]:
df['calc_agenda_points'] = df['threes'] * 3 + df['twos'] * 2 + df['ones'] + df['let_them_dream_count']

df.calc_agenda_totals == df.agenda_points

In [ ]:
df[(df['agenda_type'] == "Normal") & (df['deck_size'] == 40) & (df['average_accesses'] != 15)].head(50)